# Task 4.1 — One validated call


In [1]:
!pip install -q google-genai pydantic

In [2]:
from google import genai
from google.genai import types
from google.colab import userdata
import json
from pydantic import BaseModel, Field, ValidationError
import pandas as pd

In [3]:
SCHEMA = {   # same idea as the name / age / city example in class: properties, their types, and required
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}


In [4]:
API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY)


In [5]:
SYSTEM_TEXT = """
You are a healthcare appointment analysis assistant.

Your job is to analyse patience call_centre notes
and identify the  main reason why teh patient missed their appointment.


only use information supported by the note.
Do not invent facts or make your own words.
Keep the answer clear and make focus on the real problem.
"""

In [6]:
def create_prompt(note_text):
  prompt = f"""
  [Context] We are analysing call- centre notes of patients who missed
  their appointments.


  [Role] You are healthcare data analysis assistant.


  [Action] Read the patience note and identify the main reason why
  the patient missed the appointment.
  Allowed root_cause values:
  - forgot_or_no_reminder
  - distance_or_transport
  - cost_or_fee
  - illness_or_emergency
  - work_or_schedule
  - other

  [Tone] Be clear, professional and factual, do not use unneccessary words keep
  conversation point to point.
  Don't invent answer by yourself.


  [Format] return exactly four fileds:
  root_cause, confidence, summary, recommended_action.
  Confidence must be between 0 and 1.
  Summary must contain at most 25 words.

  PATIENT NOTE:
  {note_text}
  """
  return prompt

In [7]:
MODEL_NAME = "gemini-3.5-flash"
def analyse_note(note_text):
  prompt = create_prompt(note_text)
  response = client.models.generate_content(
      model= MODEL_NAME,
      contents=prompt,
      config=types.GenerateContentConfig(
          temperature=0,
          max_output_tokens=1000,
          response_mime_type="application/json",
          response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(
              thinking_level="minimal"
          )
      )
  )

  raw_response = response.text
  data = json.loads(response.text)
  return data, raw_response

In [8]:
N001 = "Patient forgot the appointment becuase no reminder was recieved."

result = analyse_note(N001)
print(result)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

# Task 4.2 — Validate and process the 10 notes


In [11]:
!gdown 1fMP1jWmz2-mFK4OlqlgsuiLQlehr9ENz

Downloading...
From: https://drive.google.com/uc?id=1fMP1jWmz2-mFK4OlqlgsuiLQlehr9ENz
To: /content/followup_notes.csv
100% 2.37k/2.37k [00:00<00:00, 5.19MB/s]


In [9]:
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]



class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3),
  confidence: float = Field(ge=0, le=1),
  summary: str = Field(min_length=5),
  recommended_action: str = Field(min_length=5),


def validate_root_cause(root_cause):

  if root_cause not in ALLOWED_ROOT_CAUSES:
    raise ValueError("Invalid root_case value")
  return True



validate_root_cause("forgot_or_no_reminder")


True

In [12]:
df = pd.read_csv("followup_notes.csv")
df.head(10)


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [13]:
print(df.columns)

Index(['note_id', 'appointment_id', 'note_text'], dtype='object')


In [16]:
import time
from google.genai.errors import ServerError

results = []

for index, row in df.head(10).iterrows():
  note_text = row["note_text"]

  result = {
      "note_id": row["note_id"],
      "appointment_id": row["appointment_id"],
      "root_cause": "",
      "confidence": "",
      "summary": "",
      "recommended_action": "",
      "validation_status": "",
      "raw_response": ""
  }

  if pd.isna(note_text) or str(note_text).strip() == "":
    result["root_cause"] = "other"
    result["confidence"] = 0.0
    result["summary"] = "No note recorded"
    result["recommended_action"] = "Re-call patient"
    result["validation_status"] = "skipped_blank"
  else:
    max_retries = 3
    for attempt in range(max_retries):
      try:
        data, raw_response = analyse_note(str(note_text).strip())
        result["raw_response"] = raw_response

        for field in [
            "root_cause",
            "confidence",
            "summary",
            "recommended_action"
        ]:
          if field in data:
            result[field] = data[field]

        insight = FollowupInsight(**data)
        validate_root_cause(insight.root_cause)
        result["validation_status"] = "valid"
        break
      except ServerError as e:
        print(f"API unavailable (503) for note {row['note_id']}. Retrying in 20 seconds... (Attempt {attempt+1}/{max_retries})")
        time.sleep(20)
      except ValidationError as e:
        result["validation_status"] = "invalid"
        print(f"Validation Error in note {row['note_id']}: {e}")
        break
      except Exception as e:
        result["validation_status"] = f"error: {str(e)}"
        print(f"Unexpected error in note {row['note_id']}: {e}")
        break
    else:
      result["validation_status"] = "failed_after_retries"

    if index != df.head(10).index[-1]:
      time.sleep(13)

  results.append(result)

results_df = pd.DataFrame(results)
results_df.to_csv("notes_analysed.csv", index=False)

print("Processing completed!")
print(results_df.head(10))

Processing completed!
  note_id appointment_id             root_cause  confidence  \
0    N001          A0872  forgot_or_no_reminder        0.98   
1    N002          A0897  distance_or_transport        0.95   
2    N003          A0526            cost_or_fee        0.95   
3    N004          A0926                  other        0.00   
4    N005          A0534   illness_or_emergency        0.98   
5    N006          A0507       work_or_schedule        0.98   
6    N007          A0333  forgot_or_no_reminder        0.95   
7    N008          A0222  distance_or_transport        0.95   
8    N009          A0977                  other        0.70   
9    N010          A0596       work_or_schedule        0.95   

                                             summary  \
0  The patient missed the appointment because she...   
1  Patient missed appointment due to living far f...   
2  Patient skipped the appointment due to high co...   
3                                   No note recorded   
4  P

In [18]:
# ==========================================
# TASK 4.3 — BREAK IT ON PURPOSE
# ==========================================

import json
import time
import pandas as pd
from pydantic import ValidationError
from google.genai import types

# Initialize missing global variables
if 'api_call_count' not in globals():
    api_call_count = 10  # 10 calls were made in Task 4.2 (including skipped/successful calls)

ALLOWED_CAUSES = """- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other"""

ALLOWED_SET = set(ALLOWED_ROOT_CAUSES)

# ------------------------------------------
# PART (a): Deliberately bad replies
# ------------------------------------------

print("PART (a): Testing invalid responses")
print("-----------------------------------")

# Test 1: Confidence greater than 1
bad_response_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment",
    "recommended_action": "Send an appointment reminder"
}

# Test 2: recommended_action field missing
bad_response_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.8,
    "summary": "Patient had transportation problems"
}

# Validate first bad response
try:
    insight_1 = FollowupInsight(**bad_response_1)
    print("Test 1: Response is valid")

except ValidationError as e:
    print("Test 1: Validation Error")
    print(e)


# Validate second bad response
try:
    insight_2 = FollowupInsight(**bad_response_2)
    print("Test 2: Response is valid")

except ValidationError as e:
    print("Test 2: Validation Error")
    print(e)


# ------------------------------------------
# PART (b): Compare V1 and V2 prompts
# ------------------------------------------

print("\nPART (b): Prompt comparison")
print("-----------------------------------")


# Get N007 and N009 notes from the dataset (Changed notes_df to df)
note_007 = df.loc[
    df["note_id"] == "N007", "note_text"
].iloc[0]

note_009 = df.loc[
    df["note_id"] == "N009", "note_text"
].iloc[0]


# V1: Simple prompt
def create_v1_prompt(note_text):
    return f"Classify this note: {note_text}"


# V2: CRAFT prompt with allowed categories
def create_v2_prompt(note_text):
    return f"""
[Context]
We are analysing clinic appointment no-show call-centre notes.

[Role]
You are a healthcare appointment analysis assistant.

[Action]
Identify the main reason why the patient missed the appointment.

Choose exactly one root_cause from these allowed values:
{ALLOWED_CAUSES}

If the reason is unclear or ambiguous, choose "other".
Do not guess.

[ Tone ]
Be clear, accurate and concise.

[Format]
Return only valid JSON with these four fields:
root_cause, confidence, summary, recommended_action.

The summary must contain no more than 25 words.

Patient note:
{note_text}
"""


# Function to call Gemini with the selected prompt
def compare_prompt(prompt_text):

    global api_call_count

    api_call_count += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    raw_response = response.text

    try:
        data = json.loads(raw_response)

        root_cause = data.get("root_cause", "")

        if root_cause in ALLOWED_SET:
            allowed_check = "Yes"
        else:
            allowed_check = "No"

    except (json.JSONDecodeError, AttributeError):
        data = {}
        root_cause = ""
        allowed_check = "No"

    return data, root_cause, allowed_check, raw_response


# Four prompt comparison calls
comparison_results = []

comparison_cases = [
    ("N007", str(note_007).strip(), "V1", create_v1_prompt),
    ("N007", str(note_007).strip(), "V2", create_v2_prompt),
    ("N009", str(note_009).strip(), "V1", create_v1_prompt),
    ("N009", str(note_009).strip(), "V2", create_v2_prompt)
]

for index, (note_id, note_text, version, prompt_function) in enumerate(comparison_cases):

    prompt_text = prompt_function(note_text)

    try:
        data, root_cause, allowed_check, raw_response = compare_prompt(
            prompt_text
        )

        comparison_results.append({
            "note_id": note_id,
            "prompt_version": version,
            "root_cause": root_cause,
            "allowed_value_check": allowed_check,
            "raw_response": raw_response
        })

    except Exception as e:

        comparison_results.append({
            "note_id": note_id,
            "prompt_version": version,
            "root_cause": "",
            "allowed_value_check": "No",
            "raw_response": str(e)
        })

    # Wait 13 seconds between API calls
    if index < len(comparison_cases) - 1:
        time.sleep(13)


# Display comparison table
comparison_df = pd.DataFrame(comparison_results)

print("\nPrompt Comparison Results:")
print(comparison_df[
    ["note_id", "prompt_version", "root_cause", "allowed_value_check"]
].to_string(index=False))


# ------------------------------------------
# PART (c): Total API call count
# ------------------------------------------

print("\nPART (c): Total API calls")
print("-----------------------------------")

print("Total API calls across Tasks 4.1, 4.2 and 4.3:")
print(api_call_count)

PART (a): Testing invalid responses
-----------------------------------
Test 1: Response is valid
Test 2: Response is valid

PART (b): Prompt comparison
-----------------------------------

Prompt Comparison Results:
note_id prompt_version                                                  root_cause allowed_value_check
   N007             V1 Did not receive SMS notification and forgot the appointment                  No
   N007             V2                                       forgot_or_no_reminder                 Yes
   N009             V1                                         Unreachable patient                  No
   N009             V2                                                       other                 Yes

PART (c): Total API calls
-----------------------------------
Total API calls across Tasks 4.1, 4.2 and 4.3:
14


In [28]:
# ==========================================
# TASK 4.4 — RECOMMENDATION AND GITHUB
# ==========================================

import pandas as pd
import os
import gdown

# ------------------------------------------
# PART (a): MERGE DATASETS
# ------------------------------------------

# Clean up any previously failed/empty file
if os.path.exists("appointments_clean.csv"):
    os.remove("appointments_clean.csv")

print("Downloading appointments_clean.csv using gdown...")
file_id = "1rSzDn2NpfdyaWAyYv1iAPqi5hl6ygyYg"
try:
    gdown.download(id=file_id, output="appointments_clean.csv", quiet=False)
    print("Download completed successfully!")
except Exception as e:
    print(f"Download failed: {e}")

# Read analysed notes and cleaned appointments
analysed = pd.read_csv("notes_analysed.csv")
cleaned = pd.read_csv("appointments_clean.csv")

# Merge both datasets using appointment_id
merged_notes = pd.merge(
    analysed,
    cleaned,
    on="appointment_id",
    how="inner"
)

print("Merged Dataset:")
display(merged_notes.head())

print("\nTotal merged records:", len(merged_notes))


# ------------------------------------------
# CROSS-TABULATION
# ------------------------------------------

# Compare root cause against reminder sent
crosstab_result = pd.crosstab(
    merged_notes["root_cause"],
    merged_notes["reminder_sent"]
)

print("\n--- Cross-Tabulation: Root Cause vs Reminder Sent ---")
display(crosstab_result)


# Save merged dataset
merged_notes.to_csv("merged_notes.csv", index=False)

# Save crosstab
crosstab_result.to_csv("root_cause_reminder_crosstab.csv")

print("Merged dataset and crosstab saved successfully!")


# ------------------------------------------
# PART (b): CREATE README.md
# ------------------------------------------

# IMPORTANT:
# Replace all placeholders below with your actual
# findings from Parts 1, 2, 3 and 4.

# GitHub repository link
github_link = "PASTE_YOUR_PUBLIC_GITHUB_LINK_HERE"

# Part-wise findings
part1_finding = "REPLACE_WITH_PART_1_FINDING_AND_NUMBER"
part2_finding = "REPLACE_WITH_PART_2_FINDING_AND_NUMBER"
part3_finding = "REPLACE_WITH_PART_3_FINDING_AND_NUMBER"
part4_finding = (
    f"Analysed {len(analysed)} call-centre notes and "
    f"created {len(crosstab_result)} root-cause categories in the crosstab."
)

# Model information from Part 3
selected_model = "REPLACE_WITH_SELECTED_MODEL"
model_auc = "REPLACE_WITH_ACTUAL_AUC"
decision_threshold = "REPLACE_WITH_ACTUAL_THRESHOLD"

# Three limitations selected from the assignment list
limitations = [
    "The 10 notes are an illustration and cannot show the real reasons clinic-wide.",
    "The cleaning statistics came from all rows before the split.",
    "The data are observational, so associations are not causes."
]

# Business recommendation:
# Replace the bracketed sections with actual findings and numbers.
business_recommendation = """
Based on the analysis, CareLoop should prioritise [YOUR RECOMMENDED
INTERVENTION] to reduce missed appointments. Part 1 showed [NUMBER 1
AND FINDING], while Part 2 identified [NUMBER 2 AND FINDING]. In
Part 3, the selected model achieved an AUC of [ACTUAL AUC] at a
decision threshold of [ACTUAL THRESHOLD]. Part 4 analysed 10
call-centre notes and compared the identified root causes against
reminder status. The clinic should use these findings to run a
targeted pilot, monitor its results, and evaluate whether the
intervention reduces no-shows.
"""

# Build README content
readme_content = f"""# CareLoop — Clinic Appointment No-Show Analysis

**Public GitHub Repository:** {github_link}

## Project Summary

- **Part 1:** {part1_finding}
- **Part 2:** {part2_finding}
- **Part 3:** {part3_finding}
- **Part 4:** {part4_finding}

## Business Recommendation

{business_recommendation}

## Model Selection

**Selected Model:** {selected_model}

**AUC:** {model_auc}

**Decision Threshold:** {decision_threshold}

**Why this model was selected:**

[Explain why this model was selected based on its AUC,
threshold, and the clinic's business objective.]

## Limitations

1. {limitations[0]}
2. {limitations[1]}
3. {limitations[2]}

## Project Files

This repository contains the project analysis, datasets,
notebook, and generated outputs required by the assignment.
"""

# Create README.md
with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("\nREADME.md created successfully!")

# Display README content
print("\n--- README CONTENT ---")
print(readme_content)


# ------------------------------------------
# API KEY SECURITY CHECK
# ------------------------------------------

print("\n--- API KEY SECURITY CHECK ---")

sensitive_terms = [
    "AIza",
    "careloop_clinic_key",
    "GEMINI_API_KEY"
]

for filename in os.listdir("."):

    if filename.endswith((".py", ".ipynb", ".txt", ".md", ".json", ".env")):

        try:
            with open(filename, "r", encoding="utf-8") as file:
                content = file.read()

            found_terms = [
                term for term in sensitive_terms
                if term in content
            ]

            if found_terms:
                print(
                    f"Review required: {filename} "
                    f"contains these sensitive markers: {found_terms}"
                )

        except Exception:
            print(f"Could not inspect: {filename}")

print("\nSecurity scan finished. Review flagged files before uploading.")

Downloading...
From: https://drive.google.com/uc?id=1rSzDn2NpfdyaWAyYv1iAPqi5hl6ygyYg
To: /content/appointments_clean.csv
100%|██████████| 88.1k/88.1k [00:00<00:00, 23.1MB/s]

Download completed successfully!
Merged Dataset:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,forgot_or_no_reminder,0.98,The patient missed the appointment because she...,Rebook the appointment for next week and confi...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide...",P144,C05,...,F,17.6,750.0,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,distance_or_transport,0.95,Patient missed appointment due to living far f...,Offer telehealth options or assist with coordi...,valid,"{\n ""root_cause"": ""distance_or_transport"",\n ...",P268,C07,...,M,17.2,1218.8,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment due to high co...,Provide information on available financial ass...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.95,...",P006,C04,...,F,5.3,800.0,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,NaN,P449,C04,...,M,4.5,300.0,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to a high f...,Follow up with the patient in a few days to ch...,valid,"{\n ""root_cause"": ""illness_or_emergency"",\n ...",P400,C04,...,F,4.2,550.0,App,Follow-up,Yes,0,No-Show,26,1



Total merged records: 10

--- Cross-Tabulation: Root Cause vs Reminder Sent ---


reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1


Merged dataset and crosstab saved successfully!

README.md created successfully!

--- README CONTENT ---
# CareLoop — Clinic Appointment No-Show Analysis

**Public GitHub Repository:** PASTE_YOUR_PUBLIC_GITHUB_LINK_HERE

## Project Summary

- **Part 1:** REPLACE_WITH_PART_1_FINDING_AND_NUMBER
- **Part 2:** REPLACE_WITH_PART_2_FINDING_AND_NUMBER
- **Part 3:** REPLACE_WITH_PART_3_FINDING_AND_NUMBER
- **Part 4:** Analysed 10 call-centre notes and created 6 root-cause categories in the crosstab.

## Business Recommendation


Based on the analysis, CareLoop should prioritise [YOUR RECOMMENDED
INTERVENTION] to reduce missed appointments. Part 1 showed [NUMBER 1
AND FINDING], while Part 2 identified [NUMBER 2 AND FINDING]. In
Part 3, the selected model achieved an AUC of [ACTUAL AUC] at a
decision threshold of [ACTUAL THRESHOLD]. Part 4 analysed 10
call-centre notes and compared the identified root causes against
reminder status. The clinic should use these findings to run a
targeted pilot, m